# Importing Libraries and Parameters

In [ ]:
import subprocess
import re
import sys
import os
import csv
from pynq import Overlay

VERSIONS = ["512", "768", "1024"]
BITSTREAM_DIR = "./bitstream_files" 
HW_DIR = "/home/xilinx/jupyter_notebooks/pre-mestrado/polyvec-keccak-accel-hw-sw/test"
SW_DIR = "/home/xilinx/jupyter_notebooks/pre-mestrado/kyber-ref/test"

# Jupyter on PYNQ usually runs as the 'xilinx' user.
# Accessing /dev/mem (used in hw_wrapper.c) requires root privileges.
if os.geteuid() != 0:
    print("[!] WARNING: Jupyter Notebook is not running as root.")
    print("If 'Error to open /dev/mem' occurs, start Jupyter with sudo or adjust device permissions.")
else:
    print("Root privileges detected. Access to /dev/mem granted.")

# Helper Functions

In [ ]:
def run_benchmark(executable_path):
    """Executes the test binary and extracts the results via subprocess."""
    if not os.path.exists(executable_path):
        print(f"Error: Executable not found -> {executable_path}")
        return {}
    
    print(f"Running: {executable_path} ...")
    result = subprocess.run([executable_path], capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"Execution error in {executable_path}:\n{result.stderr}")
        return {}

    return parse_output(result.stdout)

def parse_output(output):
    """Filters the test output using Regex to capture the cycles."""
    pattern = r"(.+?):\s*\nmedian:\s*(\d+)\s*cycles/ticks\naverage:\s*(\d+)\s*cycles/ticks"
    matches = re.findall(pattern, output)
    
    results = {}
    for match in matches:
        op_name = match[0].strip()
        results[op_name] = {
            "median": int(match[1]), 
            "average": int(match[2])
        }
        
    return results

def print_table(level, sw_data, hw_data):
    """Formats and prints the comparative table in the cell output."""
    print(f"\n{'='*105}")
    print(f" CONSOLIDATED RESULTS: KYBER-{level} ".center(105, "="))
    print(f"{'='*105}")
    
    header = f"| {'Operation':<32} | {'SW Median':<10} | {'HW Median':<10} | {'Spd(Med)':<9} | {'SW Average':<10} | {'HW Average':<10} | {'Spd(Avg)':<9} |"
    print(header)
    print("-" * len(header))
    
    for op in sw_data.keys():
        sw_med = sw_data[op]["median"]
        sw_avg = sw_data[op]["average"]
        
        if op in hw_data:
            hw_med = hw_data[op]["median"]
            hw_avg = hw_data[op]["average"]
            
            # Calculates the Speedup (SW Time / HW Time)
            spd_med = f"{sw_med / hw_med:.2f}x" if hw_med > 0 else "N/A"
            spd_avg = f"{sw_avg / hw_avg:.2f}x" if hw_avg > 0 else "N/A"
        else:
            hw_med = hw_avg = spd_med = spd_avg = "N/A"
            
        print(f"| {op:<32} | {sw_med:<10} | {hw_med:<10} | {spd_med:<9} | {sw_avg:<10} | {hw_avg:<10} | {spd_avg:<9} |")
    print("-" * len(header) + "\n")
    
def export_to_csv(level, sw_data, hw_data, output_dir="."):
    filename = os.path.join(output_dir, f"results_kyber_{level}.csv")
    
    with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Operation", "SW Median", "HW Median", "Spd(Med)", "SW Average", "HW Average", "Spd(Avg)"])
        
        for op in sw_data.keys():
            sw_med = sw_data[op]["median"]
            sw_avg = sw_data[op]["average"]
            
            if op in hw_data:
                hw_med = hw_data[op]["median"]
                hw_avg = hw_data[op]["average"]
                
                spd_med = f"{sw_med / hw_med:.2f}" if hw_med > 0 else "N/A"
                spd_avg = f"{sw_avg / hw_avg:.2f}" if hw_avg > 0 else "N/A"
            else:
                hw_med = hw_avg = spd_med = spd_avg = "N/A"
                
            writer.writerow([op, sw_med, hw_med, spd_med, sw_avg, hw_avg, spd_avg])
            
    print(f"[+] Results exported to: {filename}")    

# Benchmark

In [ ]:
for level in VERSIONS:
    print(f"\n>>> STARTING TEST SUITE FOR KYBER-{level} <<<")
    
    # 1. Run CPU-only version (Reference)
    sw_exec = os.path.join(SW_DIR, f"test_speed{level}")
    sw_results = run_benchmark(sw_exec)
    
    # 2. Configure FPGA for the current security level
    bitstream_path = os.path.join(BITSTREAM_DIR, f"polyvec_accel_{level}.bit")
    if os.path.exists(bitstream_path):
        print(f"Programming FPGA with {bitstream_path} ...")
        ol = Overlay(bitstream_path)
        print("FPGA successfully programmed!")
    else:
        print(f"[!] Error: Bitstream not found ({bitstream_path}). Skipping HW test.")
        continue
    
    # 3. Run HW/SW version (Accelerated)
    hw_exec = os.path.join(HW_DIR, f"test_speed{level}")
    hw_results = run_benchmark(hw_exec)
    
    # 4. Process and display the consolidated table
    if sw_results and hw_results:
        print_table(level, sw_results, hw_results)
        export_to_csv(level, sw_results, hw_results)
    else:
        print(f"Failed to consolidate results for Kyber-{level}. Check the errors above.")